In [5]:
import sys
import os
import torch
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# --- STEP 1: FIX PATHING ---
# Get the absolute path to the project root (one level up from 'notebooks')
root_path = os.path.abspath(os.path.join(os.getcwd(), "../src"))

# Add root_path to sys.path so we can find 'config' and 'src'
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# --- STEP 2: IMPORT CONFIG FIRST ---
# We import config specifically to make it available to the sub-modules
try:
    import config
    print("✅ Config file found and loaded.")
except ImportError:
    print("❌ Error: config.py not found in the root directory!")

# --- STEP 3: IMPORT AND RELOAD MODULES ---
import src.model
import src.engine
import src.preprocessing
import src.data_loader

# Force reload to apply any changes made to .py files
importlib.reload(src.model)
importlib.reload(src.engine)
importlib.reload(src.preprocessing)
importlib.reload(src.data_loader)

# --- STEP 4: IMPORT SPECIFIC FUNCTIONS ---
from src.model import SmartHealthLSTM
from src.engine import train_engine  # Use the name defined in your engine.py
from src.preprocessing import full_preprocessing_and_save

print("✅ Environment setup complete. Ready for real-world deployment.")

✅ Config file found and loaded.
✅ Environment setup complete. Ready for real-world deployment.


In [7]:
print("--- Data Dimensions ---")
print(f"X_train shape: {X_train.shape}") # Should be (Number of Patients, Features)
print(f"y_train shape: {y_train.shape}") # Should be (Number of Patients,)

# Check a small sample of the processed features
features_df = pd.DataFrame(X_train[:5]) # Show first 5 rows
print("\n--- First 5 Processed Records (Scaled) ---")
display(features_df.head())

--- Data Dimensions ---
X_train shape: (4000, 13)
y_train shape: (4000,)

--- First 5 Processed Records (Scaled) ---


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.350166,1.00451,0.809252,1.226203,-1.017147,-0.993521,-1.087630,0.680375,0.142360,-1.040312,-1.004008,-0.991536,0.734449
1,-0.095976,-0.99551,0.910658,1.226203,0.983142,-0.993521,-0.096884,1.514935,-1.023043,0.961250,0.996008,-0.991536,0.328448
2,-1.099797,1.00451,1.458251,-0.000307,0.983142,-0.993521,-1.697320,-1.452390,0.593322,0.961250,0.996008,-0.991536,-0.552948
3,0.015559,-0.99551,-0.874091,1.226203,0.983142,1.006521,0.436595,0.680375,-1.023043,0.961250,-1.004008,-0.991536,-0.399501
4,-0.542119,-0.99551,1.275720,-0.000307,0.983142,1.006521,-0.668468,-1.174203,-1.687516,-1.040312,0.996008,1.008536,0.005914


In [10]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. Ensure data is converted to PyTorch Tensors
# We use .unsqueeze(1) to change shape from (N, Features) to (N, 1, Features) for LSTM
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Define the DataLoaders (This is the part that was missing!)
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size)

print(f"✅ train_loader defined with {len(train_loader)} batches.")

✅ train_loader defined with 125 batches.


In [11]:
# Now this will NOT throw a NameError
data_iter = iter(train_loader)
batch_X, batch_y = next(data_iter)

print("--- DataLoader Inspection ---")
print(f"Batch X Shape: {batch_X.shape}") # Should be [32, 1, M]
print(f"Batch y Shape: {batch_y.shape}") # Should be [32, 1]
print(f"First label in batch: {batch_y[0].item()}")

--- DataLoader Inspection ---
Batch X Shape: torch.Size([32, 1, 13])
Batch y Shape: torch.Size([32, 1])
First label in batch: 1.0


In [15]:
import torch

# 1. Define the device locally (NOT as part of src)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize the Model using the local 'device' variable
input_dim = X_train.shape[1] 
model = SmartHealthLSTM(input_size=input_dim).to(device) # Fixed this line

# 3. Define Loss Function
criterion = nn.BCELoss()

# 4. Define Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"✅ Model initialized and moved to {device}")

✅ Model initialized and moved to cpu


In [17]:
# 4. Start the training process
print("🚀 Training in progress... Please wait.")

# We ensure 'device' is defined (lowercase)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

history = train_engine(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,  # Changed from DEVICE to device
    patience=7,    
    epochs=100     
)

print("✨ Training Complete!")

🚀 Training in progress... Please wait.
Epoch 5: Train Loss 0.0321 | Val Loss 0.0278
Epoch 10: Train Loss 0.0258 | Val Loss 0.0270
Epoch 15: Train Loss 0.0222 | Val Loss 0.0354
--- Early stopping triggered at epoch 15 ---
✨ Training Complete!


In [18]:
import os

# Create the directory if it doesn't exist
model_dir = '../models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Define the file path
model_path = os.path.join(model_dir, 'smart_health_lstm_model.pth')

# Save the model weights
torch.save(model.state_dict(), model_path)

print(f"✅ Model successfully saved to: {model_path}")

✅ Model successfully saved to: ../models\smart_health_lstm_model.pth


In [19]:
# 1. Re-initialize the model structure
loaded_model = SmartHealthLSTM(input_size=X_train.shape[1]).to(device)

# 2. Load the saved weights
loaded_model.load_state_dict(torch.load('../models/smart_health_lstm_model.pth'))

# 3. Set to evaluation mode
loaded_model.eval()

print("🚀 Model weights loaded and ready for prediction!")

🚀 Model weights loaded and ready for prediction!
